# CV14 — Ghép ảnh và văn bản

Hai encoder đưa ảnh/text về cùng không gian. Batch B cặp tạo ma trận similarity B×B; đường chéo là positive, CE theo hàng và cột. ID ngẫu nhiên chỉ kiểm tra cơ chế; dữ liệu thật phải dùng tokenizer nhất quán.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Hai encoder đưa ảnh/text về cùng không gian. Batch B cặp tạo ma trận similarity B×B; đường chéo là positive, CE theo hàng và cột. ID ngẫu nhiên chỉ kiểm tra cơ chế; dữ liệu thật phải dùng tokenizer nhất quán. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv14_open_vocab import DualEncoder, loss_fn
images = torch.rand(2, 3, 32, 32, device=device)
ids = torch.randint(0, 128, (2, 8), device=device)
mask = torch.ones(2, 8, dtype=torch.bool, device=device)
model = DualEncoder(vocab=128, d=32).to(device)
batch = (images, ids, mask)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
model.eval()
with torch.no_grad():
    similarity = model(images, ids, mask)
print('Image-to-text matrix:', similarity.shape)
print('Positive target indices:', torch.arange(len(images)).tolist())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Closed-set classifier → train dual encoder → load CLIP local → region-text alignment. Chưa có box head nên không phải open-vocabulary detector.


In [ ]:
model.eval()
with torch.no_grad():
    similarity = model(images, ids, mask)
print('Fixture retrieval choices:', similarity.argmax(-1).tolist())
print('Use load_clip(local_path) for permitted pretrained CLIP weights and its processor.')
print('A tiny randomly initialized dual encoder has no useful CLIP zero-shot ability.')


## Thử nghiệm tiếp theo

Closed-set classifier → train dual encoder → load CLIP local → region-text alignment. Chưa có box head nên không phải open-vocabulary detector.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv14_open_vocab.py) và [tài liệu gốc](../REFERENCE.md).
